In [1]:
# For tips on running notebooks in Google Colab, see
# https://docs.pytorch.org/tutorials/beginner/colab
%matplotlib inline

[Learn the Basics](intro.html) \|\|
[Quickstart](quickstart_tutorial.html) \|\|
[Tensors](tensorqs_tutorial.html) \|\| [Datasets &
DataLoaders](data_tutorial.html) \|\|
[Transforms](transforms_tutorial.html) \|\| **Build Model** \|\|
[Autograd](autogradqs_tutorial.html) \|\|
[Optimization](optimization_tutorial.html) \|\| [Save & Load
Model](saveloadrun_tutorial.html)

Build the Neural Network
========================

Neural networks comprise of layers/modules that perform operations on
data. The [torch.nn](https://pytorch.org/docs/stable/nn.html) namespace
provides all the building blocks you need to build your own neural
network. Every module in PyTorch subclasses the
[nn.Module](https://pytorch.org/docs/stable/generated/torch.nn.Module.html).
A neural network is a module itself that consists of other modules
(layers). This nested structure allows for building and managing complex
architectures easily.

In the following sections, we\'ll build a neural network to classify
images in the FashionMNIST dataset.


In [2]:
import os
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

Get Device for Training
=======================

We want to be able to train our model on an
[accelerator](https://pytorch.org/docs/stable/torch.html#accelerators)
such as CUDA, MPS, MTIA, or XPU. If the current accelerator is
available, we will use it. Otherwise, we use the CPU.


In [3]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

Using cpu device


Define the Class
================

We define our neural network by subclassing `nn.Module`, and initialize
the neural network layers in `__init__`. Every `nn.Module` subclass
implements the operations on input data in the `forward` method.


In [4]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()  # Convierte cada imagen 2D (28x28) en un vector 1D de 784 valores
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 512),  # Capa totalmente conectada: 784 entradas -> 512 neuronas
            nn.ReLU(),  # Activación no lineal que mantiene valores positivos y pone a 0 los negativos
            nn.Linear(512, 512),  # Segunda capa oculta: refina la representación aprendida
            nn.ReLU(),  # Nueva no linealidad para aumentar la capacidad del modelo
            # Capa de salida: produce 10 logits (una puntuación por clase)
            # Un logit es la salida “cruda” de la última capa lineal del modelo, puede ser cualquier número real.
            # Para convertir logits en probabilidades se aplica Softmax
            nn.Linear(512, 10),  
        )

    def forward(self, x):
        x = self.flatten(x)  # Prepara el lote de imágenes para entrar a capas lineales
        logits = self.linear_relu_stack(x)  # Ejecuta la secuencia de capas y obtiene puntuaciones sin normalizar
        return logits  # Se devuelven logits; Softmax se aplica después si se quieren probabilidades

We create an instance of `NeuralNetwork`, and move it to the `device`,
and print its structure.


In [ ]:
# Crea una instancia de la red y mueve todos sus parámetros al dispositivo seleccionado (CPU/GPU/otro acelerador)
model = NeuralNetwork().to(device)  
print(model)  # Muestra la arquitectura del modelo para verificar capas y dimensiones

NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


To use the model, we pass it the input data. This executes the model\'s
`forward`, along with some [background
operations](https://github.com/pytorch/pytorch/blob/270111b7b611d174967ed204776985cefca9c144/torch/nn/modules/module.py#L866).
Do not call `model.forward()` directly!

Calling the model on the input returns a 2-dimensional tensor with dim=0
corresponding to each output of 10 raw predicted values for each class,
and dim=1 corresponding to the individual values of each output. We get
the prediction probabilities by passing it through an instance of the
`nn.Softmax` module.


In [ ]:
X = torch.rand(1, 28, 28, device=device)  # Crea una imagen de ejemplo aleatoria (batch=1) en el mismo dispositivo del modelo.
logits = model(X)  # Pasa la entrada por la red y obtiene logits (puntuaciones crudas por clase)
pred_probab = nn.Softmax(dim=1)(logits)  # Convierte logits en probabilidades por clase; en dim=1 deben sumar 1
"""  Código equivalente a la linea anterior:
softmax = nn.Softmax(dim=1)     
pred_probab = softmax(logits)

"""
y_pred = pred_probab.argmax(1)  # Toma el índice de la probabilidad más alta como clase predicha
print(f"Predicted class: {y_pred}")

Predicted class: tensor([7])


------------------------------------------------------------------------


Model Layers
============

Let\'s break down the layers in the FashionMNIST model. To illustrate
it, we will take a sample minibatch of 3 images of size 28x28 and see
what happens to it as we pass it through the network.


In [7]:
input_image = torch.rand(3,28,28)
print(input_image.size())

torch.Size([3, 28, 28])


nn.Flatten
==========

We initialize the
[nn.Flatten](https://pytorch.org/docs/stable/generated/torch.nn.Flatten.html)
layer to convert each 2D 28x28 image into a contiguous array of 784
pixel values ( the minibatch dimension (at dim=0) is maintained).


In [8]:
flatten = nn.Flatten()
flat_image = flatten(input_image)
print(flat_image.size())

torch.Size([3, 784])


nn.Linear
=========

The [linear
layer](https://pytorch.org/docs/stable/generated/torch.nn.Linear.html)
is a module that applies a linear transformation on the input using its
stored weights and biases.


In [9]:
layer1 = nn.Linear(in_features=28*28, out_features=20)
hidden1 = layer1(flat_image)
print(hidden1.size())

torch.Size([3, 20])


nn.ReLU
=======

Non-linear activations are what create the complex mappings between the
model\'s inputs and outputs. They are applied after linear
transformations to introduce *nonlinearity*, helping neural networks
learn a wide variety of phenomena.

In this model, we use
[nn.ReLU](https://pytorch.org/docs/stable/generated/torch.nn.ReLU.html)
between our linear layers, but there\'s other activations to introduce
non-linearity in your model.


In [10]:
print(f"Before ReLU: {hidden1}\n\n")
hidden1 = nn.ReLU()(hidden1)
print(f"After ReLU: {hidden1}")

Before ReLU: tensor([[-0.3487,  0.7352, -0.5272, -0.1742, -0.3852,  0.1805,  0.4916, -0.1923,
         -0.0385, -0.0261, -0.1282, -0.6845, -0.1320,  0.4516, -0.2516, -0.3694,
          0.3941, -0.0077,  0.2553,  0.4115],
        [-0.2416,  0.1998, -0.7341,  0.1095,  0.1755,  0.1348,  0.4977, -0.3134,
         -0.0778, -0.0745,  0.1888, -0.4263,  0.3704,  0.2904, -0.4140, -0.3446,
         -0.0937,  0.1389,  0.2395,  0.1521],
        [ 0.1209,  0.4650, -0.7800,  0.3959,  0.2666,  0.1998,  0.4877, -0.1143,
         -0.2516, -0.1849, -0.4292, -0.5398,  0.2523,  0.2286, -0.6053, -0.1828,
         -0.0074, -0.1151,  0.0143,  0.1801]], grad_fn=<AddmmBackward0>)


After ReLU: tensor([[0.0000, 0.7352, 0.0000, 0.0000, 0.0000, 0.1805, 0.4916, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.4516, 0.0000, 0.0000, 0.3941, 0.0000,
         0.2553, 0.4115],
        [0.0000, 0.1998, 0.0000, 0.1095, 0.1755, 0.1348, 0.4977, 0.0000, 0.0000,
         0.0000, 0.1888, 0.0000, 0.3704, 0.2904, 0.00

nn.Sequential
=============

[nn.Sequential](https://pytorch.org/docs/stable/generated/torch.nn.Sequential.html)
is an ordered container of modules. The data is passed through all the
modules in the same order as defined. You can use sequential containers
to put together a quick network like `seq_modules`.


In [ ]:
seq_modules = nn.Sequential(  # Contenedor que aplica capas en el mismo orden en que se definen
    flatten,  # Aplana cada imagen 28x28 a un vector de 784 características
    layer1,  # Primera capa lineal: transforma 784 características en 20
    nn.ReLU(),  # Introduce no linealidad para permitir aprender patrones complejos
    nn.Linear(20, 10)  # Capa de salida: produce 10 logits (uno por clase)
 )
input_image = torch.rand(3,28,28)  # Mini-lote de 3 imágenes de ejemplo
logits = seq_modules(input_image)  # Ejecuta el paso hacia delante y obtiene logits para cada imagen

nn.Softmax
==========

The last linear layer of the neural network returns [logits]{.title-ref}
- raw values in \[-infty, infty\] - which are passed to the
[nn.Softmax](https://pytorch.org/docs/stable/generated/torch.nn.Softmax.html)
module. The logits are scaled to values \[0, 1\] representing the
model\'s predicted probabilities for each class. `dim` parameter
indicates the dimension along which the values must sum to 1.


In [13]:
softmax = nn.Softmax(dim=1)  # Crea la transformación Softmax sobre la dimensión de clases (cada fila del batch)
pred_probab = softmax(logits)  # Convierte los logits en probabilidades entre 0 y 1 que suman 1 por muestra
print(pred_probab)

tensor([[0.1119, 0.1011, 0.0994, 0.1091, 0.0776, 0.1173, 0.0968, 0.0917, 0.0894,
         0.1059],
        [0.1138, 0.1116, 0.1118, 0.1033, 0.0841, 0.1118, 0.0886, 0.0916, 0.0829,
         0.1006],
        [0.1048, 0.1023, 0.1154, 0.0917, 0.0800, 0.1089, 0.0975, 0.0923, 0.0958,
         0.1114]], grad_fn=<SoftmaxBackward0>)


Model Parameters
================

Many layers inside a neural network are *parameterized*, i.e. have
associated weights and biases that are optimized during training.
Subclassing `nn.Module` automatically tracks all fields defined inside
your model object, and makes all parameters accessible using your
model\'s `parameters()` or `named_parameters()` methods.

In this example, we iterate over each parameter, and print its size and
a preview of its values.


In [16]:
print(f"Model structure: {model}\n\n")

for name, param in model.named_parameters():
    print(f"Layer: {name} | Size: {param.size()} | Values : {param[:2]} \n")
    
print(type(name))
print(type(param))    

Model structure: NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


Layer: linear_relu_stack.0.weight | Size: torch.Size([512, 784]) | Values : tensor([[ 0.0353,  0.0038,  0.0315,  ...,  0.0066, -0.0135, -0.0286],
        [-0.0122,  0.0257,  0.0149,  ...,  0.0141, -0.0340,  0.0343]],
       grad_fn=<SliceBackward0>) 

Layer: linear_relu_stack.0.bias | Size: torch.Size([512]) | Values : tensor([-0.0039, -0.0173], grad_fn=<SliceBackward0>) 

Layer: linear_relu_stack.2.weight | Size: torch.Size([512, 512]) | Values : tensor([[ 0.0261, -0.0395, -0.0053,  ...,  0.0134, -0.0034,  0.0097],
        [ 0.0345,  0.0009,  0.0193,  ..., -0.0074,  0.0055,  0.0380]],
       grad_fn=<SliceBackward0>) 

Layer: linear_relu_stack.2.bias | 

------------------------------------------------------------------------


Further Reading
===============

-   [torch.nn API](https://pytorch.org/docs/stable/nn.html)
